# Data Exploration

Check the training data from `training/data.py` before training: depth distribution, duplicates, generation speed, and how good scramble length is as a distance label.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np

from cube.solver import solve
from cube.state import Cube
from training.data import generate

SEED = 42
N = 10_000
MAX_MOVES = 20

## Generate and view data

In [ ]:
cubes, depths = generate(N, max_moves=MAX_MOVES, seed=SEED)

for cube, depth in zip(cubes[:5], depths[:5]):
    print(depth)
    print(Cube.from_string(cube))
    print("#" * 50)

## EDA

### Depth Distribution

In [ ]:
plt.hist(depths, range(1, MAX_MOVES + 2))
plt.title("Count")
plt.xlabel("Depth")
plt.ylabel("Move Count")
plt.show()

### Duplicates

In [ ]:
for depth in range(1, MAX_MOVES + 1):
    at_depth = [cube for cube, d in zip(cubes, depths) if d == depth]
    total = len(at_depth)
    unique = len(set(at_depth))
    print(f"Unique cubes at depth {depth} => {unique} / {total}.")

### Conflicting labels

The same cube can come from scrambles of different lengths, e.g. `R L R'` gives the same cube as `L`. Those cubes appear in the data with more than one label.

In [ ]:
depths_by_cube = {}
for cube, depth in zip(cubes, depths):
    depths_by_cube.setdefault(cube, set()).add(depth)

conflicting = {cube: ds for cube, ds in depths_by_cube.items() if len(ds) > 1}
print(
    f"{len(conflicting)} of {len(depths_by_cube)} distinct cubes have more than one label."
)

for ds in list(conflicting.values())[:5]:
    print(sorted(ds))

### Generation speed

`train.py` makes fresh data every epoch, so generation must keep up with the GPU.

In [ ]:
start = time.perf_counter()
generate(N, max_moves=MAX_MOVES, seed=SEED)
elapsed = time.perf_counter() - start

print(f"{N:,} cubes in {elapsed:.2f} s ({N / elapsed:,.0f} cubes per second)")

### Label quality vs Kociemba

Kociemba solution lengths are upper bounds too (the solver returns its first solution, not the shortest), so only cubes where Kociemba is *shorter* than the label prove the label is too high. Takes about 2 minutes.

In [ ]:
SAMPLES_PER_DEPTH = 30
rng = np.random.default_rng(SEED + 1)

solve(Cube.solved())  # load the pruning tables before timing anything

solution_lengths = {}
for depth in range(1, MAX_MOVES + 1):
    at_depth = [cube for cube, d in zip(cubes, depths) if d == depth]
    sample = rng.choice(at_depth, size=SAMPLES_PER_DEPTH, replace=False)
    solution_lengths[depth] = [
        len(solve(Cube.from_string(str(cube)))) for cube in sample
    ]

In [ ]:
depth_range = list(range(1, MAX_MOVES + 1))
mean_lengths = [np.mean(solution_lengths[d]) for d in depth_range]
percent_shorter = [
    100 * np.mean([n < d for n in solution_lengths[d]]) for d in depth_range
]

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))

left.plot(depth_range, mean_lengths, marker="o", label="Kociemba solution (mean)")
left.plot(depth_range, depth_range, linestyle="--", label="Label (scramble depth)")
left.set_title("Kociemba solution length vs label")
left.set_xlabel("Scramble depth")
left.set_ylabel("Moves")
left.legend()

right.bar(depth_range, percent_shorter)
right.set_title("Cubes where Kociemba beats the label")
right.set_xlabel("Scramble depth")
right.set_ylabel("% of sample")

plt.show()

In [ ]:
for depth in depth_range:
    lengths = solution_lengths[depth]
    shorter = sum(n < depth for n in lengths)
    print(
        f"Depth {depth}: Kociemba {min(lengths)}-{max(lengths)} moves, shorter than label {shorter}/{len(lengths)}"
    )

## Conclusions

- **Depths are uniform**: about 500 cubes per depth from 1 to 20, as expected.
- **Short depths are mostly repeats**: only 18 distinct cubes at depth 1 and 43% unique at depth 2; from depth 5 on every cube is new. The model will memorise short depths, so its real test is depth 5+.
- **Conflicting labels are rare**: 43 of 9,136 distinct cubes (under 0.5%) appear with more than one label, all from short scrambles that partly cancel.
- **Generation runs at about 8,000 cubes per second** (before encoding). Fine for a first `train.py`. Data may end up being a bottleneck, will assess for now.
- **Labels overstate the distance for a sizeable minority**: at depths 4 to 13, Kociemba finds a shorter solution than the label for 10 to 17% of cubes. Beyond depth 13 this solver's solutions (18 to 25 moves) are too long to judge the labels at all, so the true error there is unknown.
- **Decision**: keep `MAX_MOVES = 20` and treat depth as a noisy label for the supervised model. Value iteration is the planned fix, and `evaluate.py` should judge the model on solve rate rather than only on matching these labels.